# Hands-on TrimNN Tutorial

Welcome to this ISMB hands-on tutorial for [**TrimNN**](https://github.com/yuyang-0825/TrimNN), a tool for analyzing cellular community motifs in spatial omics data.

TrimNN starts from a table of cells with spatial coordinates and cell-type labels. It converts those cells into a graph, then uses neural models to query or discover recurring cellular community motifs.

By the end of this tutorial, you will know how to:

- Prepare spatial omics data for TrimNN.
- Construct a cellular community graph from cell coordinates.
- Query a user-defined motif.
- Discover enriched motifs automatically.
- Visualize motif locations on the original tissue coordinates.
- Adapt the workflow to your own spatial dataset.

> Workshop note: this notebook is designed to be safe to open and inspect first. Commands that run TrimNN are controlled by `RUN_TRIMNN_COMMANDS` in the setup cell.


## 1. Workflow Overview

**Learning objective:** understand the full TrimNN pipeline before running commands.

TrimNN works because it turns spatial cell neighborhoods into graph objects. In the graph, **nodes are cells** and **edges connect neighboring cells**. Motifs are small connected graph patterns with cell-type labels.

```mermaid
flowchart LR
    A[Raw spatial data CSV] --> B[csv2gml.py]
    B --> C[Cellular community graph GML]
    C --> D[Function 1 Query user-defined motif]
    C --> E[Function 2 Discover motifs of one size]
    C --> F[Function 3 Discover motifs across sizes]
    D --> G[Predicted motif occurrence]
    E --> H[Top enriched motifs]
    F --> H
    C --> I[Visualization]
    I --> J[Motif locations on spatial map]
```

**Input:** a CSV file with at least `X`, `Y`, and `cell_type` columns.

**Output:** one or more GML graph files plus result tables and motif visualizations.


## 2. Environment Setup

**Learning objective:** configure paths and decide whether this notebook should execute TrimNN commands.

Why this step matters: TrimNN depends on PyTorch, DGL, igraph, and other packages. During a workshop, it is useful to inspect the notebook first and only run the command-line steps once the environment is ready.

**Expected input:** a working Python environment, ideally Python 3.9 as recommended by the upstream TrimNN README.

**Expected output:** printed paths showing the notebook workspace, Python executable, and TrimNN checkout path.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

WORKDIR = Path.cwd()
REPO_URL = "https://github.com/yuyang-0825/TrimNN.git"
REPO_DIR = WORKDIR / "TrimNN"

# Keep False for teaching/demo mode. Set to True when dependencies are installed.
RUN_TRIMNN_COMMANDS = True


def run_command(cmd, cwd=REPO_DIR, run=RUN_TRIMNN_COMMANDS, check=True):
    """Print a command and optionally run it with live output."""
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ cd {cwd}\n$ {printable}")
    if not run:
        print("Skipped because RUN_TRIMNN_COMMANDS is False.")
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {printable}")
    return result


def prune_args(use_prune):
    """Return CLI args for pruning. Omit -prune for False because upstream argparse uses type=bool."""
    return ["-prune", "True"] if use_prune else []

print(f"Notebook workspace: {WORKDIR}")
print(f"Python executable: {sys.executable}")
print(f"TrimNN checkout path: {REPO_DIR}")
print(f"RUN_TRIMNN_COMMANDS: {RUN_TRIMNN_COMMANDS}")


### Recommended installation commands

Run these in a terminal, not inside the notebook.

```bash
conda create -n TrimNNEnv python=3.9
conda activate TrimNNEnv
```

CPU-only PyTorch examples from the upstream README:

```bash
# Linux / Windows CPU-only
conda install pytorch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1 cpuonly -c pytorch

# macOS CPU-only
conda install pytorch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1 -c pytorch
```

Then install DGL and the remaining requirements:

```bash
pip install dgl==1.1.2 -f https://data.dgl.ai/wheels/repo.html
pip install -r requirements.txt
```

If you are using a CUDA Linux machine, use the CUDA-specific PyTorch and DGL commands from the TrimNN README.


In [ ]:
# Lightweight dependency check. This cell does not install packages.
packages = ["torch", "dgl", "networkx", "pandas", "scipy", "sklearn", "igraph", "matplotlib", "seaborn"]
for package in packages:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "installed")
        print(f"{package}: {version}")
    except Exception as exc:
        print(f"{package}: not available ({exc.__class__.__name__}: {exc})")


## 3. Get TrimNN

**Learning objective:** obtain the TrimNN source code and demo data.

Why this step matters: the tutorial calls scripts from the TrimNN repository, including `csv2gml.py`, `TrimNN.py`, `visualize.py`, and `vf2_analysis.py`.

**Input:** internet access for the first clone, or an existing local `TrimNN/` directory.

**Output:** a `TrimNN/` folder containing scripts, `demo_data/`, and the pretrained model.


In [ ]:
if REPO_DIR.exists():
    print(f"Found existing repository: {REPO_DIR}")
else:
    run_command(["git", "clone", REPO_URL, str(REPO_DIR)], cwd=WORKDIR, run=RUN_TRIMNN_COMMANDS)

if REPO_DIR.exists():
    print("Repository files:")
    for path in sorted(REPO_DIR.iterdir()):
        print("-", path.name)
else:
    print("TrimNN repository is not available yet. Clone it or set REPO_DIR to an existing checkout.")


## 4. Explore the Demo Dataset

**Learning objective:** inspect the spatial cell table that will become a graph.

Why this step matters: graph construction only makes sense if the input coordinates and cell-type labels are correct. TrimNN expects one row per cell.

**Input:** `TrimNN/demo_data/demo_data.csv`

Required columns:

- `X`: cell x-coordinate
- `Y`: cell y-coordinate
- `cell_type`: cell-type label

**Output:** a table preview, cell-type counts, and a scatter plot of cells in spatial coordinates.


In [ ]:
import pandas as pd

demo_csv = REPO_DIR / "demo_data" / "demo_data.csv"
df = pd.read_csv(demo_csv)

required_columns = {"X", "Y", "cell_type"}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print(f"Demo data shape: {df.shape}")
display(df.head())
display(df["cell_type"].value_counts().rename_axis("cell_type").reset_index(name="n_cells"))
print("Cell types:", ", ".join(sorted(df["cell_type"].unique())))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Match the color assignment used by TrimNN/visualize.py.
unique_cell_types = df["cell_type"].unique()
palette = sns.color_palette("hls", len(unique_cell_types))
color_map = dict(zip(unique_cell_types, palette))

fig, ax = plt.subplots(figsize=(7, 6))
for cell_type in unique_cell_types:
    group = df[df["cell_type"] == cell_type]
    ax.scatter(group["X"], group["Y"], s=8, alpha=0.8, label=cell_type, color=color_map[cell_type])
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Demo spatial cells by cell type")
ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()


**Expected output:** the demo dataset should contain hundreds of cells and multiple cell types.

**Interpretation:** each point is a cell. The color corresponds to `cell_type`. Spatially nearby cells may become connected in the cellular community graph.


## 5. Build the Cellular Community Graph

**Learning objective:** convert cell coordinates into a graph that TrimNN can analyze.

Why this step matters: TrimNN does not directly analyze a CSV table. It analyzes a graph in which cells are nodes and spatial neighbors are edges. The upstream `csv2gml.py` script builds these edges with Delaunay triangulation.

**Input:** `demo_data/demo_data.csv`

**Output:**

- `demo_data/demo_data.gml`: cellular community graph
- `demo_data/cell_type_to_id.csv`: mapping from cell-type names to integer labels

Important note: We include an optional edge-pruning step for noise reduction. It removes unusually long edges, which are often artifacts near tissue boundaries. Specifically, we compute the length of every edge and use the 99th percentile as the cutoff. To enable this step, simply set `USE_PRUNE` to True


In [ ]:
USE_PRUNE = False

target_gml = REPO_DIR / "demo_data" / "demo_data.gml"

run_command([
    sys.executable, "csv2gml.py",
    "-target", "demo_data/demo_data.csv",
    "-out", "demo_data/demo_data.gml",
    *prune_args(USE_PRUNE),
])

print(f"Expected graph file: {target_gml}")


In [ ]:
# Inspect the generated mapping between original cell types and integer labels.
mapping_csv = REPO_DIR / "demo_data" / "cell_type_to_id.csv"
if mapping_csv.exists():
    cell_type_map = pd.read_csv(mapping_csv)
    display(cell_type_map)
else:
    print("Run the csv2gml step to generate cell_type_to_id.csv")


### Visualize the Delaunay triangulation graph

The same Delaunay triangulation idea is used by `csv2gml.py` to define spatial neighbors. This quick plot helps you inspect whether the graph structure looks reasonable before running TrimNN.

**Input:** cell coordinates and cell-type labels from the demo CSV.

**Output:** a spatial graph view where grey lines are Delaunay edges and colored points are cells.


In [ ]:
import numpy as np
from scipy.spatial import Delaunay

# Match TrimNN/visualize.py: grey Delaunay edges plus cell-type colors from the hls palette.
unique_cell_types = df["cell_type"].unique()
palette = sns.color_palette("hls", len(unique_cell_types))
color_map = dict(zip(unique_cell_types, palette))

points = np.stack((df["X"], df["Y"]), axis=-1)
tri = Delaunay(points)

fig, ax = plt.subplots(figsize=(7, 6))
ax.triplot(points[:, 0], points[:, 1], tri.simplices, color="grey", linewidth=0.25, alpha=0.5)
for cell_type in unique_cell_types:
    group = df[df["cell_type"] == cell_type]
    ax.scatter(group["X"], group["Y"], s=8, alpha=0.9, label=cell_type, color=color_map[cell_type])
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Delaunay triangulation graph used for cellular communities")
ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()


**Interpretation:** each grey line is cell-cell neighborhood edge generated by Delaunay triangulation. Dense regions produce many short edges, while boundary regions can create longer edges. If long boundary edges dominate your own dataset, try graph pruning.


**Expected output:** after a successful run, `demo_data.gml` and `cell_type_to_id.csv` should exist.

**Interpretation:** the GML file stores the graph. Each node is a cell, each edge connects neighboring cells, and each node label is the integer ID of a cell type.


## 6. Choose a TrimNN Function

**Learning objective:** decide which TrimNN analysis mode matches your question before running the commands.

TrimNN provides three analysis functions. Start by choosing whether you want to test one motif, discover motifs of a fixed size, or search across multiple sizes.

| Function | Purpose |
| --- | --- |
| Function 1 | Query a user-defined motif |
| Function 2 | Discover enriched motifs of one size |
| Function 3 | Discover enriched motifs across multiple sizes |

After this overview, the tutorial walks through Function 1, Function 2, and Function 3 in order.


## 7. Function 1: subgraph matching

**Learning objective:** search for a motif chosen by the user.

Why this step matters: Function 1 is useful when you already have a biological pattern of interest, such as repeated neighborhoods involving a particular cell type combination.

**Input:**

- Target graph: `demo_data/demo_data.gml`
- Motif graph: `demo_data/size-3.gml`

**Output:** predicted occurrence count for the input motif.

The example query motif in this tutorial is `CTX-Ex_CTX-Ex_CTX-Ex`: a size-3 triangle in which all three neighboring cells are labeled CTX-Ex.


In [ ]:
# Simple illustration of the CTX-Ex_CTX-Ex_CTX-Ex size-3 motif used in Function 1.
fig, ax = plt.subplots(figsize=(5.2, 4.2))
coords = {0: (0, 0), 1: (1.4, 0), 2: (0.7, 1.15)}
edges = [(0, 1), (1, 2), (2, 0)]
labels = {0: "CTX-Ex", 1: "CTX-Ex", 2: "CTX-Ex"}

for u, v in edges:
    ax.plot([coords[u][0], coords[v][0]], [coords[u][1], coords[v][1]], color="#555555", linewidth=2, zorder=1)
for node, (x, y) in coords.items():
    ax.scatter(x, y, s=1100, color="#88c0d0", edgecolor="#333333", linewidth=1.5, zorder=3)
    ax.text(x, y, labels[node], ha="center", va="center", fontsize=10, zorder=4)
ax.set_title("CTX-Ex_CTX-Ex_CTX-Ex motif", pad=14)
ax.set_xlim(-0.45, 1.85)
ax.set_ylim(-0.35, 1.55)
ax.set_axis_off()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


In [ ]:
motif_size = 3
motif_label = "CTX-Ex_CTX-Ex_CTX-Ex"

available_cell_types = set(df["cell_type"].unique())
motif_parts = motif_label.split("_")
unknown_labels = sorted(set(motif_parts).difference(available_cell_types))
if unknown_labels:
    raise ValueError(f"Motif label contains unknown cell types: {unknown_labels}")
if len(motif_parts) != motif_size:
    raise ValueError(f"motif_size={motif_size}, but motif_label contains {len(motif_parts)} labels")

run_command([
    sys.executable, "csv2gml.py",
    "-target", "demo_data/demo_data.csv",
    "-out", "demo_data/demo_data.gml",
    "-motif_size", str(motif_size),
    "-motif_label", motif_label,
    *prune_args(USE_PRUNE),
])

motif_gml = REPO_DIR / "demo_data" / f"size-{motif_size}.gml"
print(f"Expected motif file: {motif_gml}")


In [ ]:
run_command([
    sys.executable, "TrimNN.py",
    "-function", "subgraph_matching",
    "-motif", "demo_data/size-3.gml",
    "-k", "2",
    "-target", "demo_data/demo_data.gml",
    "-outpath", "result_function1/",
])


In [ ]:
result_dir = REPO_DIR / "result_function1"
if result_dir.exists():
    print("Function 1 outputs:")
    for path in sorted(result_dir.iterdir()):
        print("-", path.name)
        if path.suffix.lower() in {".csv", ".txt"}:
            print(path.read_text()[:1000])
else:
    print("Run Function 1 to create result_function1/")


**Expected output:** `result_function1/` should contain a text result with the predicted number of motif occurrences.

**Interpretation:** a larger predicted count suggests the queried cellular community motif appears more often in the graph. Function 1 answers a targeted question rather than discovering motifs automatically.


## 8. Function 2: top overrepresented motifs of a specific size

**Learning objective:** find top motifs of a fixed size without specifying the motif label in advance.

Why this step matters: Function 2 is useful for exploratory analysis. Instead of asking about one known motif, it searches candidate motifs of a chosen size and reports high-scoring mootifs.


**Input:** target graph and motif size.

**Output:** CSV table of predicted occurrence values plus a GML file for overrepresented motifs.


In [ ]:
n_cell_types = df["cell_type"].nunique()
print(f"Number of cell types in demo CSV: {n_cell_types}")

run_command([
    sys.executable, "TrimNN.py",
    "-function", "specific_size",
    "-size", "3",
    "-k", "2",
    "-target", "demo_data/demo_data.gml",
    "-celltype", str(n_cell_types),
    "-outpath", "result_function2/",
])


In [ ]:
result_dir = REPO_DIR / "result_function2"
if result_dir.exists():
    print("Function 2 outputs:")
    for path in sorted(result_dir.iterdir()):
        print("-", path.name)
    csv_outputs = sorted(result_dir.glob("*.csv"))
    if csv_outputs:
        function2_df = pd.read_csv(csv_outputs[0])
        sort_column = "predicted_occurrence_number"
        if sort_column in function2_df.columns:
            display(function2_df.sort_values(sort_column, ascending=False).head(10))
        else:
            print(f"Column '{sort_column}' was not found. Available columns: {list(function2_df.columns)}")
            display(function2_df.head(10))
else:
    print("Run Function 2 to create result_function2/")


**Expected output:** `result_function2/` should include a predicted occurrence CSV and an overrepresented motif GML file.

**Interpretation:** Use the cell-type mapping file to translate integer labels back to biological cell-type names.


## 9. Function 3: top overrepresented motifs across sizes

**Learning objective:** discover enriched motifs while allowing motif size to vary.

Why this step matters: real cellular communities may involve more than one neighborhood scale. Function 3 grows motifs from size 3 up to the requested maximum size.

**Input:** target graph, maximum motif size, search strategy.

**Output:** predicted occurrence tables and motif GML files for multiple motif sizes.


In [ ]:
run_command([
    sys.executable, "TrimNN.py",
    "-function", "all_size",
    "-size", "4",
    "-k", "2",
    "-target", "demo_data/demo_data.gml",
    "-celltype", str(n_cell_types),
    "-outpath", "result_function3/",
    "-search", "greedy",
])


In [ ]:
result_dir = REPO_DIR / "result_function3"
if result_dir.exists():
    print("Function 3 outputs:")
    for path in sorted(result_dir.iterdir()):
        print("-", path.name)
    for csv_path in sorted(result_dir.glob("*.csv")):
        print()
        print(f"Top predicted motifs in: {csv_path.name}")
        function3_df = pd.read_csv(csv_path)
        sort_column = "predicted_occurrence_number"
        if sort_column in function3_df.columns:
            display(function3_df.sort_values(sort_column, ascending=False).head(5))
        else:
            print(f"Column '{sort_column}' was not found. Available columns: {list(function3_df.columns)}")
            display(function3_df.head(5))
else:
    print("Run Function 3 to create result_function3/")


**Expected output:** one or more predicted occurrence CSV files, usually one per motif size.

**Interpretation:** compare results across sizes to see whether enriched communities are mostly small local motifs or larger multi-cell structures.


## 10. Visualize Motif Locations

**Learning objective:** map motif occurrences back onto spatial coordinates.

Why this step matters: motif counts are useful, but spatial interpretation requires seeing where motif instances occur in the tissue.

**Input:** original CSV, motif size, motif label.

**Output:** a figure in `visualization/` showing highlighted motif instances.


In [ ]:
visual_motif_size = 3
visual_motif_label = "CTX-Ex_CTX-Ex_CTX-Ex"

run_command([
    sys.executable, "visualize.py",
    "-target", "demo_data/demo_data.csv",
    "-outpath", "visualization/",
    "-motif_size", str(visual_motif_size),
    "-motif_label", visual_motif_label,
])


In [ ]:
from IPython.display import Image, display

viz_dir = REPO_DIR / "visualization"
if viz_dir.exists():
    print("Visualization outputs:")
    for path in sorted(viz_dir.iterdir()):
        print("-", path.name)
        if path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
            display(Image(filename=str(path)))
else:
    print("Run the visualization step to create visualization/")


**Expected output:** a PNG figure such as `size-3_visualization.png`.

**Interpretation:** grey lines show the background Delaunay graph. Colored points show cell types. Highlighted blue edges mark places where the selected motif appears. Clusters of highlighted motifs may indicate spatially repeated cellular communities.


## 11. Apply TrimNN to Your Own Dataset

**Learning objective:** adapt the demo workflow to a new spatial omics dataset.

Use this checklist before running TrimNN on your own data.

### Step 1: Prepare CSV

Your CSV should contain one row per cell or spot and at least these columns:

- `X`
- `Y`
- `cell_type`

Recommended for the current upstream scripts: place your CSV under `TrimNN/demo_data/` and use a relative path such as `demo_data/my_spatial_cells.csv`. Some upstream path handling assumes simple relative paths.

### Step 2: Convert CSV to GML

Run `csv2gml.py` to create a graph.

### Step 3: Run TrimNN

Choose Function 1, 2, or 3 depending on your question.

### Step 4: Visualize Results

For motifs of size 1 to 3, use `visualize.py` to map motif locations back to the spatial coordinates.


In [ ]:
# Template for your own dataset.
# Put the file under TrimNN/demo_data/ first, then edit this relative path.
my_csv = Path("demo_data/my_spatial_cells.csv")
my_out_gml = Path("demo_data/my_spatial_cells.gml")

my_csv_abs = REPO_DIR / my_csv
if my_csv_abs.exists():
    my_df = pd.read_csv(my_csv_abs)
    required = {"X", "Y", "cell_type"}
    missing = required.difference(my_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    my_df["cell_type"] = my_df["cell_type"].astype(str).str.strip()
    print(my_df.shape)
    print(f"Unique cell types: {my_df['cell_type'].nunique()}")
    display(my_df.head())

    run_command([
        sys.executable, "csv2gml.py",
        "-target", str(my_csv),
        "-out", str(my_out_gml),
    ])
else:
    print("Edit my_csv to point to your data before running this template.")


**Expected output:** a new GML file for your dataset.

**Interpretation:** once the graph exists, the same Function 1, 2, and 3 commands can be reused by replacing `demo_data/demo_data.gml` with your new GML path and updating `-celltype`.


## 12. Mini Challenges

Use these exercises during the workshop to make the workflow interactive.

1. **Try another motif.** Change `motif_label` from `CTX-Ex_CTX-Ex_CTX-Ex` to another size-3 combination using existing cell types.
2. **Change motif size.** Try Function 2 with `-size 4`. Watch how runtime and output files change.
3. **Compare pruning.** Run graph construction with `USE_PRUNE = False`, then with `USE_PRUNE = True`, and compare the resulting graph or motif outputs.
4. **Interpret a result table.** Pick one top motif from Function 2 and translate its integer labels using `cell_type_to_id.csv`.
5. **Visualize a different motif.** Update `visual_motif_label` and rerun the visualization section.


## 13. Take-home Messages

By the end of this tutorial, keep these points in mind:

- TrimNN turns spatial cell coordinates and cell-type annotations into graph representations that can be queried for biologically meaningful local neighborhoods.
- The three TrimNN functions support complementary motif analyses: testing a user-defined motif, discovering enriched motifs of a fixed size, and exploring motifs across multiple sizes.
- Motif results are most useful when linked back to the original spatial map and translated into interpretable cell-type patterns.


## 14. Troubleshooting

- `ModuleNotFoundError: dgl`: install the DGL build matching your PyTorch and CUDA/CPU setup.
- `ModuleNotFoundError: igraph` or `python-igraph` installation errors: try `conda install -c conda-forge python-igraph` or install system build tools before pip installation.
- `undefined symbol: iJIT_NotifyEvent`: the upstream README suggests installing `mkl==2024.0`.
- Wrong Python executable: rerun the setup cell and confirm `sys.executable` points to your TrimNN environment.
- Missing `demo_data/demo_data.gml`: rerun the graph construction section.
- Missing `cell_type_to_id.csv`: rerun `csv2gml.py` and confirm the output folder is writable.
- Motif label error: check spelling, capitalization, and whitespace in `cell_type` values.
- Very long runtime: start with size 3 and the demo data, then increase motif size only after the pipeline works.
- Too many nodes in a k-hop subgraph: try pruning, reduce `-k`, or inspect whether the graph has unusually long edges.
